# Fiche 7 — Joindre des tables : INNER JOIN, LEFT JOIN

| | |
|---|---|
| 🎯 **Objectif** | Relier plusieurs tables et prévoir combien de lignes une jointure va produire |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 6 |
| 🔗 **Dans le projet** | Partie 3 (requêtes qui croisent plusieurs tables) |

> **L'idée clé.** Une jointure recolle ce que la normalisation a séparé. Avant de l'écrire, il faut savoir **combien de lignes** on attend : c'est le meilleur détecteur d'erreur.

In [1]:
import sqlite3

import pandas as pd

# Ouvre TA base si elle est prête ; sinon, la base de démonstration du dossier (demande-la à ton mentor
# si elle ne t'a pas été transmise). Ouverture en lecture seule : les exemples ne peuvent rien abîmer.
base = sqlite3.connect("file:../data/processed/dataimmo.sqlite?mode=ro", uri=True)


def sql(requete: str) -> pd.DataFrame:
    return pd.read_sql(requete, base)

sql("SELECT COUNT(*) AS nb_biens FROM bien")

,nb_biens
0,34169


## 1. INNER JOIN … ON : ne garder que ce qui correspond

Chaque bien porte la clé de sa commune. `INNER JOIN … ON` ajoute à chaque bien les colonnes de sa commune, en
reliant les deux clés :

In [2]:
sql("""
SELECT b.id_bien, b.type_local, c.nom_commune, c.code_departement
FROM bien AS b
INNER JOIN commune AS c ON c.id_codedep_codecommune = b.id_codedep_codecommune
LIMIT 5
""")

,id_bien,type_local,nom_commune,code_departement
0,1,Appartement,Chevry,01
1,2,Appartement,Antibes,06
2,3,Appartement,Nice,06
3,4,Appartement,Saint-Laurent-du-Var,06
4,5,Appartement,Aubagne,13


`AS b` et `AS c` sont des **alias** de table : `b.type_local` se lit « la colonne type_local de la table bien ».

**Prévoir le nombre de lignes** : chaque bien a exactement une commune, la jointure doit donc garder tous les biens.

In [3]:
sql("""
SELECT COUNT(*) AS lignes_apres_jointure
FROM bien AS b
INNER JOIN commune AS c ON c.id_codedep_codecommune = b.id_codedep_codecommune
""")

,lignes_apres_jointure
0,34169


## 2. ⚠️ Piège : la jointure qui fait disparaître des lignes

Imaginons que les communes de Corse aient été perdues à l'import (code « 2A » mal lu). On le simule en ajoutant une
condition à la jointure :

In [4]:
sql("""
SELECT COUNT(*) AS biens_gardes
FROM bien AS b
INNER JOIN commune AS c ON c.id_codedep_codecommune = b.id_codedep_codecommune
                       AND c.code_departement NOT IN ('2A', '2B')
""")

,biens_gardes
0,33918


251 biens ont disparu, **sans aucune erreur**. Le `LEFT JOIN` garde, lui, **tous** les biens de la table de gauche ;
ceux qui n'ont pas de correspondance ont des colonnes de commune vides (`NULL`). C'est l'outil pour **trouver** les
lignes orphelines :

In [5]:
sql("""
SELECT b.id_codedep_codecommune, COUNT(*) AS biens_orphelins
FROM bien AS b
LEFT JOIN commune AS c ON c.id_codedep_codecommune = b.id_codedep_codecommune
                      AND c.code_departement NOT IN ('2A', '2B')
WHERE c.id_codedep_codecommune IS NULL
GROUP BY b.id_codedep_codecommune
ORDER BY biens_orphelins DESC
LIMIT 5
""")

,id_codedep_codecommune,biens_orphelins
0,2A004,104
1,2A247,21
2,2A249,15
3,2A130,14
4,2A139,12


## 3. Enchaîner les jointures en suivant les clés

Pour passer d'une commune à sa région : commune → region. Pour passer d'une vente à sa région : vente → bien →
commune → region, une clé étrangère à chaque étape.

### ✍️ À toi de jouer

Pour chaque **région** : le nombre de communes présentes dans la base et leur population totale, triés par nombre de communes décroissant. **Avant d'exécuter** : quel total de communes (toutes régions confondues) attends-tu ?

In [6]:
# Ta réponse ici : sql(""" ... """)

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```sql
SELECT
    r.nom_region,
    COUNT(*)          AS nb_communes,
    SUM(c.population) AS population_totale
FROM commune AS c
INNER JOIN region AS r ON r.id_region = c.id_region
GROUP BY r.id_region, r.nom_region
ORDER BY nb_communes DESC;
```

Le total de la colonne `nb_communes` doit être égal au nombre de lignes de `commune` (3 125 dans la base de démonstration) : sinon, la jointure a perdu des communes. On regroupe par `id_region` **et** `nom_region` : la clé garantit qu'on ne fusionne pas deux régions de même nom.
</details>

## ✅ À retenir

1. `INNER JOIN … ON` ne garde que les lignes qui ont une correspondance ; `LEFT JOIN` garde toute la table de gauche.
2. Toujours prévoir le nombre de lignes attendu, puis le vérifier avec `COUNT(*)`.
3. Une jointure qui perd des lignes ne produit **aucune erreur** : seul le comptage la révèle.
4. `LEFT JOIN … WHERE droite.cle IS NULL` trouve les orphelins.